
<a id='rag'></a>

# Retrieval-Augmented Generation (RAG)

## RAG på norsk: Gjenfinningsforsterket tekstgenerering


## Språkmodellen

Vi kommer til å bruke modeller fra [Ollama](https://ollama.com/), en kjent plattform for modeller som kan brukes både på lokal maskin og i skyløsninger. I denne oppgaven vil vi bruke LLM [gemma3:1b](https://ollama.com/library/gemma3), som er en familie av modeller fra Google DeepMind. Dette er en liten modell med bare 1 milliard parametere. Det bør være mulig å bruke den på de fleste bærbare maskiner.

In [ ]:
pwd

In [ ]:
import os
print(os.getcwd())
print(output_folder)

In [ ]:
import os
# os.environ['HF_HOME'] = ''~/sti/til/dine/dokumenter''
# vi må ha os på grunn av tokenet fra HF

import torch
device = 0 if torch.cuda.is_available() else -1

from langchain_community.llms import Ollama

llm = Ollama(
    model="mistral:latest"
)

query = 'What are the major contributions of the Trivandrum Observatory?'
output = llm.invoke(query)
print(output)

In [ ]:
from langchain_ollama import OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(
    model="granite-embedding:latest",
)

document_folder = '/Users/ragnhildsundsbak/rtd-litteratur'

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader

# Print length of documents before embedding
documents = []
for filename in os.listdir(document_folder):
    if filename.endswith(".pdf"):
        path = os.path.join(document_folder, filename)
        loader = PyPDFLoader(path)
        documents.extend(loader.load())

print(len(documents))
print(documents[0].metadata)
print(documents[0].page_content[:500])

print(f'Number of documents:', len(documents))
print('Maximum document length: ', max([len(doc.page_content) for doc in documents]))

In [ ]:
# Print one of the documents
print(documents[0])

In [ ]:
# Chunking and embedding Printing new length, this time of chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700, #  Could be more, for larger models like mistralai/Ministral-8B-Instruct-2410
    chunk_overlap  = 200,
)
documents = text_splitter.split_documents(documents)

print(f'Antall dokuemnt-chunks etter splitting:', len(documents))
print('Maximum document length: ', max([len(doc.page_content) for doc in documents]))

In [ ]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents, ollama_embeddings)

In [ ]:
# første gang man lager embedding:
#import os
#import torch

#from langchain_community.document_loaders import PyPDFLoader
#from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain_ollama import OllamaEmbeddings
#from langchain_community.vectorstores import FAISS

# Embeddings fra Ollama
ollama_embeddings = OllamaEmbeddings(
    model="granite-embedding:latest",
)

# 1. Last alle PDF-er fra mappen
document_folder = "/Users/ragnhildsundsbak/rtd-litteratur"

# 3. Bygg FAISS-vektorstore fra dokumenter (embedder + indekserer)
print("Genererer embeddings og oppretter FAISS-indeks...")
vectorstore = FAISS.from_documents(documents, ollama_embeddings)

# 4. Lagre FAISS-indeksen lokalt
output_folder = "data/embeddings"  # velg selv hvor i prosjektet
os.makedirs(output_folder, exist_ok=True)

print(f"Lagrer FAISS-indeksen til mappen: {output_folder}/")
vectorstore.save_local(output_folder)
print("Lagring fullført!")


In [ ]:
# Senere, legger til nye PDFer

#import os

#from langchain_community.document_loaders import PyPDFLoader
#from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain_ollama import OllamaEmbeddings
#from langchain_community.vectorstores import FAISS

# Samme embedding-modell som før
ollama_embeddings = OllamaEmbeddings(
    model="granite-embedding:latest",
)

# 1. Last eksisterende FAISS-indeks fra disk
output_folder = "data/embeddings"

print(f"Laster eksisterende FAISS-indeks fra: {output_folder}/")
vectorstore = FAISS.load_local(
    output_folder,
    ollama_embeddings,
    allow_dangerous_deserialization=True,  # ofte nødvendig i nyere langchain-versjoner
)
print("Indeks lastet!")

# 2. Finn nye PDF-er (her må du selv definere hva som er "nytt":
#    - annen mappe,
#    - eller holde en liste/logg over allerede behandlede filer).
document_folder = "/Users/ragnhildsundsbak/rtd-tutorial/data/pdf/new_pdfs"

new_documents = []
for filename in os.listdir(document_folder):
    if filename.endswith(".pdf"):
        path = os.path.join(document_folder, filename)
        # Her kunne du filtrert bort filer du vet er med fra før
        loader = PyPDFLoader(path)
        new_documents.extend(loader.load())

print(f"Nye sider (før splitting): {len(new_documents)}")

# 3. Splitt de nye dokumentene i chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=200,
)
new_documents = text_splitter.split_documents(new_documents)

print(f"Nye dokument-chunks (etter splitting): {len(new_documents)}")

# 4. Legg til de nye chunksene i eksisterende indeks (append)
print("Legger til nye dokumenter i eksisterende FAISS-indeks...")
vectorstore.add_documents(new_documents)
print("Ferdig med å legge til.")

# 5. Lagre indeksen på nytt (samme mappe/filnavn)
print(f"Lagrer oppdatert FAISS-indeks til: {output_folder}/")
vectorstore.save_local(output_folder)
print("Oppdatert indeks lagret.")


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

In [ ]:
from langchain_classic.prompts import PromptTemplate

prompt_template = '''You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
Context: {context}

Question: {input}

Answer:
'''

prompt = PromptTemplate(template=prompt_template,
                        input_variables=['context', 'input'])

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

combine_documents_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_documents_chain)

result = rag_chain.invoke({'input': query})

print(result['answer'])

Får du feilmeldinger? Lant ned [Sublime text](https://www.sublimetext.com/download) slik at du lettere får oversikt over koden din. Output filen gir et linjenummer for der feilen ligger. Da kan du lese av rette linjenummeret i Sublime editoren.